In [5]:

import chess

def get_legal_moves_for_state(move_sequence):
    """
    Replays a game from a list of SAN moves and returns the 
    legal moves available at that specific board state.
    """
    board = chess.Board()
    
    # Replay the move sequence
    for move in move_sequence:
        board.push_san(move)
        
    # Get all legal moves in SAN format (e.g., 'Nf3', 'axb5', 'O-O')
    # We use board.san(move) to convert the move object to text
    legal_moves_san = [board.san(move) for move in board.legal_moves]
    
    return board, legal_moves_san

# --- Test Case 1: Ruy Lopez (Open Game) - 11 Moves Deep (5.5 turns) ---
moves_ruy_lopez = [
    "e4", "e5", 
    "Nf3", "Nc6", 
    "Bb5", "a6", 
    "Ba4", "Nf6", 
    "O-O", "Be7", 
    "Re1"
]

# --- Test Case 2: Sicilian Defense (Semi-Open) - 12 Moves Deep (6 turns) ---
moves_sicilian = [
    "e4", "c5", 
    "Nf3", "d6", 
    "d4", "cxd4", 
    "Nxd4", "Nf6", 
    "Nc3", "a6", 
    "Bg5", "e6"
]

# --- Test Case 3: Queen's Gambit Declined (Closed) - 10 Moves Deep (5 turns) ---
moves_qgd = [
    "d4", "d5", 
    "c4", "e6", 
    "Nc3", "Nf6", 
    "Bg5", "Be7", 
    "e3", "O-O"
]

# List of cases to iterate through
test_cases = [
    ("Ruy Lopez", moves_ruy_lopez),
    ("Sicilian Defense", moves_sicilian),
    ("Queen's Gambit", moves_qgd)
]

# Execute and Print Results
for name, moves in test_cases:
    board, legal_moves = get_legal_moves_for_state(moves)
    
    print(f"")
    print(f"Last Move Played: {moves[-1]}")
    print("Current Board State:")
    print(board)
    print(f"Legal Moves: {legal_moves}...")
    print("-" * 40 + "\n")


Last Move Played: Re1
Current Board State:
r . b q k . . r
. p p p b p p p
p . n . . n . .
. . . . p . . .
B . . . P . . .
. . . . . N . .
P P P P . P P P
R N B Q R . K .
Legal Moves: ['Rg8', 'Rf8', 'Kf8', 'Rb8', 'Ra7', 'Bf8', 'Bd6', 'Bc5', 'Bb4', 'Ba3', 'Ng8', 'Nh5', 'Nd5', 'Ng4', 'Nxe4', 'Nb8', 'Na7', 'Na5', 'Nd4', 'Nb4', 'O-O', 'h6', 'g6', 'd6', 'b6', 'a5', 'h5', 'g5', 'd5', 'b5']...
----------------------------------------


Last Move Played: e6
Current Board State:
r n b q k b . r
. p . . . p p p
p . . p p n . .
. . . . . . B .
. . . N P . . .
. . N . . . . .
P P P . . P P P
R . . Q K B . R
Legal Moves: ['Bh6', 'Bxf6', 'Bh4', 'Bf4', 'Be3', 'Bd2', 'Bc1', 'Nxe6', 'Nc6', 'Nf5', 'Ndb5', 'Nf3', 'Nb3', 'Nde2', 'Nd5', 'Ncb5', 'Na4', 'Nce2', 'Nb1', 'Rg1', 'Bxa6', 'Bb5+', 'Bc4', 'Bd3', 'Be2', 'Ke2', 'Kd2', 'Qh5', 'Qg4', 'Qf3', 'Qd3', 'Qe2', 'Qd2', 'Qc1', 'Qb1', 'Rc1', 'Rb1', 'e5', 'h3', 'g3', 'f3', 'b3', 'a3', 'h4', 'g4', 'f4', 'b4', 'a4']...
----------------------------------------


Las

In [6]:
import chess
import json
import numpy as np
from decimal import Decimal
from sentence_transformers import SentenceTransformer

def generate_templates(board, move, san) -> list[str]:
    piece = board.piece_at(move.from_square)
    target = chess.square_name(move.to_square)
    is_capture = board.is_capture(move)
    is_castle = board.is_castling(move)
    is_promotion = move.promotion is not None
    templates = []
    file_name = chess.FILE_NAMES[chess.square_file(move.from_square)]

    if is_castle:
        if chess.square_file(move.to_square) == 6:
            templates += ["Castle kingside", "Kingside castle", "Perform kingside castling",
                          "King castles short", "Castle short", "Short castle"]
        else:
            templates += ["Castle queenside", "Queenside castle", "Perform queenside castling",
                          "King castles long", "Castle long", "Long castle"]
        return templates

    if is_promotion:
        promo_piece = chess.piece_name(move.promotion)
        templates += [f"Promote pawn to {promo_piece} on {target}",
                      f"Advance pawn to {target} and promote to {promo_piece}",
                      f"Move pawn to {target} and make it a {promo_piece}"]
        return templates

    p_name = chess.piece_name(piece.piece_type)
    if is_capture:
        templates += [f"{p_name.capitalize()} captures on {target}",
                      f"Capture the piece on {target} with the {p_name}",
                      f"Take on {target} using the {p_name}",
                      f"Use the {p_name} to capture on {target}"]
    elif piece.piece_type == chess.PAWN:
        templates += [f"Advance pawn to {target}", f"Push the pawn to {target}",
                      f"Move pawn to {target}", f"Advance {file_name} pawn"]
    else:
        templates += [f"Move the {p_name} to {target}", f"Play {p_name} to {target}",
                      f"Develop the {p_name} to {target}", f"{p_name.capitalize()} goes to {target}"]
    templates.append(f"Play {san}")
    return templates

def get_move_descriptions(board: chess.Board) -> list[dict]:
    """
    Takes a board, returns all (san, desc) pairs for every legal move —
    the format your embedding pipeline expects.
    """
    records = []
    for move in board.legal_moves:
        san = board.san(move)
        for desc in generate_templates(board, move, san):
            records.append({"move": san, "desc": desc})
    return records


EMBEDDING_MODELS = {
    "MiniLM-L6":   "all-MiniLM-L6-v2",
    "mpnet-base":  "all-mpnet-base-v2",
    "bge-small":   "BAAI/bge-small-en-v1.5",
}

loaded_models = {}
for short_name, model_id in EMBEDDING_MODELS.items():
    print(f"Loading {model_id}...")
    loaded_models[short_name] = SentenceTransformer(model_id)
print(f"All {len(loaded_models)} embedding models loaded ✓")

model = loaded_models["MiniLM-L6"]


def build_index(records: list[dict], embed_model=None) -> tuple[list, np.ndarray]:
    """
    Embeds all descriptions and returns:
      - the original records list  (so you can look up move_san by index)
      - a matrix of embeddings    (shape: N x embedding_dim)
    """
    if embed_model is None:
        embed_model = model
    descriptions = [r["desc"] for r in records]
    embeddings = embed_model.encode(descriptions, convert_to_numpy=True)
    return records, embeddings


def find_best_move(user_query: str, records: list[dict], embeddings: np.ndarray, embed_model=None) -> str:
    if embed_model is None:
        embed_model = model
    query_vec = embed_model.encode(user_query, convert_to_numpy=True)
    norms = np.linalg.norm(embeddings, axis=1) * np.linalg.norm(query_vec)
    scores = embeddings @ query_vec / norms
    best_idx = int(np.argmax(scores))
    return records[best_idx]["move"]


def process_board_state(board: chess.Board, user_query: str) -> str:
    """
    All-in-one function Thomas can call in his game loop:
      board       – the current chess.Board object
      user_query  – whatever the user typed

    Returns the matched SAN move string (e.g. "Nf3", "O-O", "exd5").
    """
    records = get_move_descriptions(board)     
    records, embeddings = build_index(records) 
    return find_best_move(user_query, records, embeddings)


c:\Users\thoma\Downloads\175\.conda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 136.08it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading all-mpnet-base-v2...


Loading weights: 100%|██████████| 199/199 [00:01<00:00, 136.17it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading BAAI/bge-small-en-v1.5...


Loading weights: 100%|██████████| 199/199 [00:01<00:00, 135.28it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All 3 embedding models loaded ✓


In [7]:
# ── Test: simulate one turn ──────────────────────────────────────────────────
test_board = chess.Board()  # standard starting position

print("Current Board:")
print(test_board)
print()

test_queries = [
    "push the pawn in the center",
    "bring out the knight on the right",
    "move a pawn forward",
    "develop a piece",
]

for query in test_queries:
    matched = process_board_state(test_board, query)
    print(f"Query:  '{query}'")
    print(f"Matched: {matched}")
    print()

Current Board:
r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R

Query:  'push the pawn in the center'
Matched: b4

Query:  'bring out the knight on the right'
Matched: Nc3

Query:  'move a pawn forward'
Matched: a3

Query:  'develop a piece'
Matched: Nc3



In [8]:
import chess
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading {LLM_MODEL_ID} tokenizer + model (this may take a minute)...")
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
print(f"{LLM_MODEL_ID} loaded ✓")

SMOL_MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

print(f"Loading {SMOL_MODEL_ID} tokenizer + model...")
smol_tokenizer = AutoTokenizer.from_pretrained(SMOL_MODEL_ID)
# Use float32 — if the GPU is full from Qwen, device_map="auto" places this
# model on CPU where float16 silently produces garbage / repeated tokens.
smol_model = AutoModelForCausalLM.from_pretrained(
    SMOL_MODEL_ID,
    torch_dtype=torch.float32,
    device_map="auto",
)
print(f"{SMOL_MODEL_ID} loaded ✓  (device: {next(smol_model.parameters()).device})")


def _llm_pick_move_with(tokenizer, model, user_query: str, board: chess.Board) -> str:
    """
    Generic helper: ask any causal LLM to choose the best SAN move.
    Returns a SAN string (or '' if parsing fails).
    """
    legal_sans = sorted([board.san(m) for m in board.legal_moves])
    fen = board.fen()

    prompt = (
        f"You are a chess assistant. Given the board position (FEN) and a list of legal moves, "
        f"pick the single best move that matches the user's intent.\n\n"
        f"FEN: {fen}\n"
        f"Legal moves: {', '.join(legal_sans)}\n"
        f"User says: \"{user_query}\"\n\n"
        f"Reply with ONLY the move in standard algebraic notation (SAN), nothing else."
    )

    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    raw = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    cleaned = raw.split()[0] if raw else ""
    if cleaned in legal_sans:
        return cleaned
    for san in legal_sans:
        if san in raw:
            return san
    return cleaned


def llm_pick_move(user_query: str, board: chess.Board) -> str:
    """Qwen 2.5 1.5B wrapper."""
    return _llm_pick_move_with(llm_tokenizer, llm_model, user_query, board)


def smol_pick_move(user_query: str, board: chess.Board) -> str:
    """SmolLM2 1.7B wrapper."""
    return _llm_pick_move_with(smol_tokenizer, smol_model, user_query, board)


_board = chess.Board()
_test = llm_pick_move("push the king's pawn forward two squares", _board)
print(f"Qwen picked: {_test}")

_test2 = smol_pick_move("push the king's pawn forward two squares", _board)
print(f"SmolLM2 picked: {_test2}")


Loading Qwen/Qwen2.5-1.5B-Instruct tokenizer + model (this may take a minute)...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:08<00:00, 42.12it/s, Materializing param=model.norm.weight]                               


Qwen/Qwen2.5-1.5B-Instruct loaded ✓
Loading HuggingFaceTB/SmolLM2-1.7B-Instruct tokenizer + model...


Loading weights: 100%|██████████| 218/218 [00:14<00:00, 14.99it/s, Materializing param=model.norm.weight]                              
Some parameters are on the meta device because they were offloaded to the cpu and disk.


HuggingFaceTB/SmolLM2-1.7B-Instruct loaded ✓  (device: cpu)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Qwen picked: a5
SmolLM2 picked: Nf3


In [9]:
import json
import chess
from pathlib import Path


def validate_sample(sample):
    test_id = sample["id"]
    premoves = sample.get("premoves", [])
    instruction = sample.get("instruction")
    expected = sample.get("expected")
    fen_override = sample.get("note")

    # Start board
    if fen_override:
        try:
            board = chess.Board(fen_override)
        except Exception as e:
            return False, f"Invalid FEN: {fen_override}"
    else:
        board = chess.Board()

        # Apply premoves
        for mv in premoves:
            try:
                board.push_san(mv)
            except Exception:
                return False, f"Illegal premove '{mv}' in position {board.fen()}"

    # Check expected move
    legal_sans = [board.san(m) for m in board.legal_moves]

    if expected not in legal_sans:
        return False, (
            f"Expected move '{expected}' not legal.\n"
            f"Position: {board.fen()}\n"
            f"Legal moves: {legal_sans}"
        )

    return True, "OK"

valid_samples = []
def validate_dataset(folder):
    all_files = list(Path(folder).glob("*.json"))
    total = 0
    valid = 0

    for file in all_files:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, dict):
            data = [data]

        for sample in data:
            total += 1
            ok, msg = validate_sample(sample)

            if ok:
                valid += 1
                valid_samples.append(sample)
            else:
                print(f"\n❌ Sample {sample['id']} invalid")
                print(msg)

    print("\nSummary")
    print("--------")
    print(f"Valid: {valid}")
    print(f"Total: {total}")
    print(f"Invalid: {total-valid}")


# Run validator
validate_dataset("samples")
output_file = "valid_chess_samples.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(valid_samples, f, indent=2, ensure_ascii=False)

print(f"Saved {len(valid_samples)} valid samples to {output_file}")


❌ Sample 4 invalid
Expected move 'Bxc4' not legal.
Position: rnbqkbnr/ppp1pppp/8/8/2pP4/8/PP2PPPP/RNBQKBNR w KQkq - 0 3
Legal moves: ['Nh3', 'Nf3', 'Kd2', 'Qa4+', 'Qd3', 'Qb3', 'Qd2', 'Qc2', 'Bh6', 'Bg5', 'Bf4', 'Be3', 'Bd2', 'Nc3', 'Na3', 'Nd2', 'd5', 'h3', 'g3', 'f3', 'e3', 'b3', 'a3', 'h4', 'g4', 'f4', 'e4', 'b4', 'a4']

❌ Sample 5 invalid
Expected move 'axb4' not legal.
Position: rnbqk2r/pppp1ppp/4pn2/8/1bPP4/2N5/PP2PPPP/R1BQKBNR w KQkq - 2 4
Legal moves: ['Nh3', 'Nf3', 'Kd2', 'Qa4', 'Qd3', 'Qb3', 'Qd2', 'Qc2', 'Bh6', 'Bg5', 'Bf4', 'Be3', 'Bd2', 'Rb1', 'd5', 'c5', 'h3', 'g3', 'f3', 'e3', 'b3', 'a3', 'h4', 'g4', 'f4', 'e4', 'a4']

❌ Sample 13 invalid
Expected move 'Nxb4' not legal.
Position: rnbqk1nr/ppp2ppp/4p3/3p4/1bPP4/2N5/PP2PPPP/R1BQKBNR w KQkq - 2 4
Legal moves: ['Nh3', 'Nf3', 'Kd2', 'Qa4+', 'Qd3', 'Qb3', 'Qd2', 'Qc2', 'Bh6', 'Bg5', 'Bf4', 'Be3', 'Bd2', 'Rb1', 'cxd5', 'c5', 'h3', 'g3', 'f3', 'e3', 'b3', 'a3', 'h4', 'g4', 'f4', 'e4', 'a4']

❌ Sample 22 invalid
Expected move 'B

In [11]:

import chess
import numpy as np
from collections import defaultdict
from pathlib import Path
import os

def load_testcases_from_json(json_path):
    """
    Expected JSON format:
    {
        "id": number,
        "premoves": [],
        "instruction": "...",
        "expected": "...",
        "note": "FEN or None"
    }
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Handle both single-object and list-of-objects JSON
    if isinstance(data, dict):
        data = [data]

    test_cases = []
    for entry in data:
        test_id = entry.get("id")
        premoves = entry.get("premoves", [])
        instruction = entry.get("instruction")
        expected = entry.get("expected")
        fen_override = entry.get("note")  # this stores FEN or None

        test_cases.append((test_id, premoves, instruction, expected, fen_override))

    return test_cases

def load_multiple_jsons(folder_path):
    all_cases = []

    for file in Path(folder_path).glob("*.json"):
        all_cases.extend(load_testcases_from_json(file))

    return all_cases

#TEST_CASES = load_testcases_from_json("valid_samples.json")
TEST_CASES = [
    (1, [], "advance the king pawn two squares", "e4", None),
    (2, [], "play the queen pawn two squares", "d4", None),
    (3, [], "develop the king knight", "Nf3", None),
    (4, [], "move the queen bishop pawn", "c4", None),
    (5, ['e4'], "push the d pawn two squares", "d4", None),
    (6, ['e4', 'e5'], "develop the king knight toward the center", "Nf3", None),
    (7, ['e4', 'c5'], "develop the king knight", "Nf3", None),
    (8, ['d4', 'd5'], "develop the queen knight", "Nc3", None),
    (9, ['d4', 'Nf6'], "play the queen pawn support move", "c4", None),
    (10, ['c4'], "develop the king knight", "Nf3", None),

    (11, ['e4', 'e5', 'Nf3', 'Nc6'], "develop the bishop to b5", "Bb5", None),
    (12, ['e4', 'c5', 'Nf3'], "strike the center with d pawn", "d4", None),
    (13, ['d4', 'd5', 'c4'], "accept the gambit", "dxc4", None),
    (14, ['d4', 'Nf6', 'c4', 'g6'], "develop the knight", "Nc3", None),
    (15, ['e4', 'e6'], "play the d pawn two squares", "d4", None),
    (16, ['e4', 'c6'], "advance the d pawn", "d4", None),
    (17, ['e4', 'e5', 'Nf3', 'Nc6', 'Bc4'], "develop the bishop aggressively", "Bc5", None),
    (18, ['d4', 'd5', 'Nf3'], "defend the pawn with e pawn", "e6", None),
    (19, ['c4', 'e5'], "capture the pawn", "dxe5", None),
    (20, ['Nf3', 'd5'], "strike the center", "d4", None),

    (21, ['e4', 'e5', 'Nf3', 'Nc6', 'Bb5'], "attack the e4 pawn", "a6", None),
    (22, ['e4', 'c5', 'Nf3', 'd6'], "develop the d pawn", "d4", None),
    (23, ['d4', 'Nf6', 'c4', 'e6'], "develop the queen knight", "Nc3", None),
    (24, ['e4', 'd5'], "capture the pawn on d5", "exd5", None),
    (25, ['e4', 'e5', 'Bc4'], "develop the knight", "Nc6", None),
    (26, ['d4', 'd5', 'c4', 'e6'], "develop the knight to c3", "Nc3", None),
    (27, ['e4', 'c5', 'Nc3'], "develop the knight", "Nc6", None),
    (28, ['Nf3', 'Nf6'], "play g3 to fianchetto", "g3", None),
    (29, ['c4', 'c5'], "develop the knight", "Nc3", None),
    (30, ['d4', 'g6'], "play the c pawn forward", "c4", None),

    (31, ['e4', 'e5', 'Nf3', 'Nc6', 'Bc4', 'Bc5'], "castle kingside", "O-O", None),
    (32, ['e4', 'c5', 'Nf3', 'Nc6'], "play d4", "d4", None),
    (33, ['d4', 'Nf6', 'c4', 'g6', 'Nc3'], "develop bishop to g7", "Bg7", None),
    (34, ['e4', 'e5', 'Nf3', 'd6'], "play d4", "d4", None),
    (35, ['d4', 'd5', 'c4', 'c6'], "develop knight", "Nc3", None),
    (36, ['e4', 'c6', 'd4', 'd5'], "develop knight", "Nc3", None),
    (37, ['c4', 'e6', 'Nc3'], "play d5", "d5", None),
    (38, ['Nf3', 'd5', 'g3'], "develop bishop", "Bg4", None),
    (39, ['e4', 'e5', 'Qh5'], "defend the pawn", "Nc6", None),
    (40, ['e4', 'e5', 'Bc4', 'Nc6'], "attack f7", "Qh5", None),

    (41, ['d4', 'Nf6', 'c4', 'e5'], "capture pawn", "dxe5", None),
    (42, ['e4', 'c5', 'Nf3', 'd6', 'd4'], "capture on d4", "cxd4", None),
    (43, ['e4', 'e5', 'Nf3', 'Nc6', 'Bb5', 'a6'], "retreat bishop", "Ba4", None),
    (44, ['e4', 'c5', 'c3'], "develop knight", "Nf6", None),
    (45, ['d4', 'd5', 'Bf4'], "develop knight", "Nf6", None),
    (46, ['c4', 'g6', 'Nc3'], "develop bishop", "Bg7", None),
    (47, ['Nf3', 'c5'], "play e4", "e4", None),
    (48, ['e4', 'd6', 'd4'], "develop knight", "Nf6", None),
    (49, ['d4', 'Nf6', 'Bg5'], "play e6", "e6", None),
    (50, ['e4', 'e5', 'Nf3', 'Nc6', 'd4'], "capture pawn", "exd4", None),

    (51, ['e4', 'c5', 'Nf3', 'd6', 'd4', 'cxd4'], "recapture with knight", "Nxd4", None),
    (52, ['d4', 'd5', 'c4', 'dxc4'], "recover pawn", "e3", None),
    (53, ['e4', 'e5', 'Nf3', 'Nc6', 'Bc4', 'Nf6'], "castle kingside", "O-O", None),
    (54, ['c4', 'e5', 'Nc3', 'Nf6'], "push d pawn", "d4", None),
    (55, ['Nf3', 'd5', 'g3', 'c5'], "develop bishop", "Bg2", None),
    (56, ['e4', 'e6', 'd4', 'd5'], "develop knight", "Nc3", None),
    (57, ['d4', 'Nf6', 'c4', 'c5'], "play d5", "d5", None),
    (58, ['e4', 'c5', 'Nf3', 'Nc6', 'd4'], "capture pawn", "cxd4", None),
    (59, ['e4', 'e5', 'Nc3'], "develop knight", "Nc6", None),
    (60, ['d4', 'd5', 'Nc3'], "develop knight", "Nf6", None),

    (61, ['e4', 'e5', 'Nf3', 'Nc6', 'Bb5', 'Nf6'], "castle kingside", "O-O", None),
    (62, ['c4', 'e6', 'Nc3', 'd5'], "play d4", "d4", None),
    (63, ['Nf3', 'Nf6', 'c4'], "play g6", "g6", None),
    (64, ['e4', 'c5', 'Nc3', 'Nc6'], "play f4", "f4", None),
    (65, ['d4', 'Nf6', 'Bg5', 'e6'], "play e4", "e4", None),
    (66, ['c4', 'g6', 'g3'], "develop bishop", "Bg7", None),
    (67, ['Nf3', 'd5', 'e3'], "develop bishop", "Bg4", None),
    (68, ['e4', 'e5', 'f4'], "capture pawn", "exf4", None),
    (69, ['d4', 'd5', 'Bf4', 'Nf6'], "play e3", "e3", None),
    (70, ['e4', 'c5', 'Nf3', 'e6'], "play d4", "d4", None),

    (71, ['e4', 'c5', 'Nf3', 'd6', 'Bb5+'], "block with bishop", "Bd7", None),
    (72, ['d4', 'Nf6', 'c4', 'g6', 'Nc3', 'Bg7'], "play e4", "e4", None),
    (73, ['e4', 'e5', 'Nf3', 'Nc6', 'Bc4', 'Bc5', 'c3'], "develop knight", "Nf6", None),
    (74, ['c4', 'c6'], "play d4", "d4", None),
    (75, ['Nf3', 'g6', 'g3'], "develop bishop", "Bg7", None),
    (76, ['e4', 'd5', 'exd5'], "recapture with queen", "Qxd5", None),
    (77, ['d4', 'Nf6', 'c4', 'e6', 'Nc3', 'Bb4'], "play e3", "e3", None),
    (78, ['e4', 'c5', 'b4'], "capture pawn", "cxb4", None),
    (79, ['c4', 'e5', 'g3'], "develop knight", "Nc6", None),
    (80, ['Nf3', 'c5', 'e4'], "develop knight", "Nc6", None),

    (81, ['e4', 'e5', 'Nf3', 'Nc6', 'd4', 'exd4'], "capture with knight", "Nxd4", None),
    (82, ['d4', 'd5', 'c4', 'c6', 'Nc3'], "develop knight", "Nf6", None),
    (83, ['e4', 'c5', 'Nf3', 'Nc6', 'Bb5'], "play g6", "g6", None),
    (84, ['c4', 'g6', 'Nc3', 'Bg7'], "play d4", "d4", None),
    (85, ['Nf3', 'd5', 'd4'], "capture pawn", "Nf6", None),
    (86, ['e4', 'e5', 'Bc4', 'Bc5', 'Qh5'], "defend f7", "Qe7", None),
    (87, ['d4', 'Nf6', 'c4', 'g6', 'g3'], "develop bishop", "Bg7", None),
    (88, ['e4', 'c5', 'Nc3', 'd6'], "play f4", "f4", None),
    (89, ['c4', 'e6', 'g3'], "play d5", "d5", None),
    (90, ['Nf3', 'Nf6', 'c4', 'g6'], "develop bishop", "Bg2", None),

    (91, ['e4', 'e5', 'Nf3', 'Nc6', 'Bb5', 'a6', 'Ba4'], "develop knight", "Nf6", None),
    (92, ['d4', 'd5', 'c4', 'e6', 'Nc3', 'Be7'], "develop knight", "Nf3", None),
    (93, ['e4', 'c5', 'Nf3', 'd6', 'd4', 'cxd4', 'Nxd4'], "develop knight", "Nf6", None),
    (94, ['c4', 'e5', 'Nc3', 'Nc6'], "play g3", "g3", None),
    (95, ['Nf3', 'd5', 'g3', 'Nf6'], "develop bishop", "Bg2", None),
    (96, ['e4', 'e6', 'd4', 'd5', 'Nc3'], "pin the knight", "Bb4", None),
    (97, ['d4', 'Nf6', 'c4', 'e6', 'Nf3'], "play b6", "b6", None),
    (98, ['e4', 'c5', 'Nf3', 'Nc6', 'd4', 'cxd4', 'Nxd4', 'g6'], "develop bishop", "Be3", None),
    (99, ['c4', 'g6', 'Nc3', 'Bg7', 'd4'], "play d6", "d6", None),
    (100, ['e4', 'c5', 'Nf3', 'd6', 'Bc4'], "develop knight", "Nf6", None),
]

def claude_pick_move(user_query: str, board: chess.Board) -> str:
    """Claude Sonnet 3.5 API wrapper using Anthropic"""
    try:
        import anthropic
    except ImportError:
        return "N/A (install anthropic)"

    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        return "N/A (no API key)"

    client = anthropic.Anthropic(api_key=api_key)
    legal_sans = sorted([board.san(m) for m in board.legal_moves])
    fen = board.fen()
    
    prompt = (
        f"You are a chess assistant. Given the board position (FEN) and a list of legal moves, "
        f"pick the single best move that matches the user's intent.\n\n"
        f"FEN: {fen}\n"
        f"Legal moves: {', '.join(legal_sans)}\n"
        f"User says: \"{user_query}\"\n\n"
        f"Reply with ONLY the move in standard algebraic notation (SAN), nothing else. Do not add punctuation."
    )
    
    try:
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=16,
            temperature=0,
            messages=[{"role": "user", "content": prompt}]
        )
        raw = response.content[0].text.strip()
        cleaned = raw.split()[0] if raw else ""
        if cleaned in legal_sans:
            return cleaned
        for san in legal_sans:
            if san in raw:
                return san
        return cleaned
    except Exception as e:
        return f"Error: {e}"

LLM_MODELS = {
    "Claude Sonnet 3.5": claude_pick_move,
    "Qwen 1.5B":  llm_pick_move,
    "SmolLM2 1.7B": smol_pick_move,
}

def find_top_k(user_query, records, embeddings, embed_model, k=3):
    query_vec = embed_model.encode(user_query, convert_to_numpy=True)
    norms = np.linalg.norm(embeddings, axis=1) * np.linalg.norm(query_vec)
    scores = embeddings @ query_vec / (norms + 1e-9)
    top_indices = np.argsort(scores)[::-1]
    seen, results = set(), []
    for idx in top_indices:
        san = records[idx]["move"]
        if san not in seen:
            seen.add(san)
            results.append((san, float(scores[idx])))
        if len(results) >= k:
            break
    return results

boards_and_records = {}
for test_id, move_seq, query, expected, fen_override in TEST_CASES:
    if fen_override:
        board = chess.Board(fen_override)
    else:
        board = chess.Board()
        for mv in move_seq:
                board.push_san(mv)
    legal_sans = [board.san(m) for m in board.legal_moves]
    if expected not in legal_sans:
        boards_and_records[test_id] = None
        continue
    records = get_move_descriptions(board)
    boards_and_records[test_id] = (board, records, legal_sans)

embed_names = list(loaded_models.keys())
llm_names   = list(LLM_MODELS.keys())
col_w = 14
header_models = "".join(f"{n:<{col_w}}" for n in embed_names)
header_llms   = "".join(f"{n:<{col_w}}" for n in llm_names)

print("=" * 140)
print(f"  CHESS NL→SAN EVALUATION  |  {len(embed_names)} Embedding Models + {len(llm_names)} LLMs  |  {len(TEST_CASES)} tests")
print("=" * 140)
print(f"  {'#':<4} {'Query':<36} {header_models}{header_llms}{'Exp':<8}")
print("-" * 140)

scores_top1 = defaultdict(int)
scores_top3 = defaultdict(int)
llm_correct = defaultdict(int)
total = 0
all_results = []

for test_id, move_seq, query, expected, fen_override in TEST_CASES:
    entry = boards_and_records[test_id]
    if entry is None:
        print(f"  {test_id:<4} [SKIP – expected move not legal]")
        continue

    board, records, legal_sans = entry
    total += 1

    row_preds = {}
    row_ok    = {}

    # ── Each embedding model ──
    for name, emb_model in loaded_models.items():
        recs, embs = build_index(records, embed_model=emb_model)
        top3 = find_top_k(query, recs, embs, emb_model, k=3)
        top3_sans = [s for s, _ in top3]
        pred = top3_sans[0] if top3_sans else "—"
        ok = pred == expected
        t3 = expected in top3_sans
        if ok: scores_top1[name] += 1
        if t3: scores_top3[name] += 1
        row_preds[name] = pred
        row_ok[name] = ok

    # ── Each LLM ──
    for llm_name, llm_fn in LLM_MODELS.items():
        pred = llm_fn(query, board)
        ok = pred == expected
        if ok: llm_correct[llm_name] += 1
        row_preds[llm_name] = pred
        row_ok[llm_name] = ok

    # ── Print row ──
    q_display = (query[:33] + "…") if len(query) > 34 else query
    cells = ""
    for name in embed_names:
        mark = "✓" if row_ok[name] else "✗"
        cells += f"{row_preds[name]:<11}{mark}  "
    for llm_name in llm_names:
        mark = "✓" if row_ok[llm_name] else "✗"
        cells += f"{row_preds[llm_name]:<11}{mark}  "
    print(f"  {test_id:<4} {q_display:<36} {cells}{expected}")

    all_results.append({
        "id": test_id, "query": query, "expected": expected,
        "preds": {**row_preds},
        "correct": {**row_ok},
    })

# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 140)
print(f"\n  RESULTS SUMMARY  (n={total})\n")
print(f"  {'Model':<20} {'Top-1':>8} {'Top-3':>8} {'Top-1 %':>10}")
print(f"  {'-'*20} {'-'*8} {'-'*8} {'-'*10}")
for name in embed_names:
    t1 = scores_top1[name]
    t3 = scores_top3[name]
    print(f"  {name:<20} {t1:>5}/{total}  {t3:>5}/{total}  {t1/total*100:>8.1f}%")
for llm_name in llm_names:
    lc = llm_correct[llm_name]
    print(f"  {llm_name:<20} {lc:>5}/{total}  {'—':>8}  {lc/total*100:>8.1f}%")
print()

# ── Best embedding vs best LLM ───────────────────────────────────────────────
best_embed_name = max(embed_names, key=lambda n: scores_top1[n])
best_embed_acc  = scores_top1[best_embed_name] / total * 100
best_llm_name   = max(llm_names, key=lambda n: llm_correct[n])
best_llm_acc    = llm_correct[best_llm_name] / total * 100
print(f"  Best embedding: {best_embed_name} ({best_embed_acc:.1f}%)  vs  Best LLM: {best_llm_name} ({best_llm_acc:.1f}%)  "
      f"Δ = {best_embed_acc - best_llm_acc:+.1f} pp")

# ── Per-model misses ──────────────────────────────────────────────────────────
print()
for name in embed_names + llm_names:
    misses = [r for r in all_results if not r["correct"][name]]
    if misses:
        print(f"  {name} misses:")
        for r in misses:
            print(f"    #{r['id']:>2}  Got '{r['preds'][name]}' expected '{r['expected']}'")
    else:
        print(f"  {name}: perfect ✓")

print("=" * 140)


  CHESS NL→SAN EVALUATION  |  3 Embedding Models + 3 LLMs  |  100 tests
  #    Query                                MiniLM-L6     mpnet-base    bge-small     Claude Sonnet 3.5Qwen 1.5B     SmolLM2 1.7B  Exp     
--------------------------------------------------------------------------------------------------------------------------------------------
  1    advance the king pawn two squares    a4         ✗  a4         ✗  a4         ✗  N/A (install anthropic)✗  a5         ✗  e4         ✓  e4
  2    play the queen pawn two squares      a4         ✗  a4         ✗  b4         ✗  N/A (install anthropic)✗  a4         ✗  Nf3        ✗  d4
  3    develop the king knight              Nc3        ✗  Nc3        ✗  Nc3        ✗  N/A (install anthropic)✗  Nh3        ✗  Nf3        ✓  Nf3
  4    move the queen bishop pawn           b4         ✗  b4         ✗  b4         ✗  N/A (install anthropic)✗  a5         ✗  Nf3        ✗  c4
  5    [SKIP – expected move not legal]
  6    develop the king knight tow